# 第15章 回测原理与陷阱

> 本 notebook 为《量化投资》配套代码，用内置数据离线可跑。

In [ ]:
# Colab 自举：在 Colab 上自动克隆仓库并安装 qfin（本地用 uv 运行会自动跳过）
import importlib.util, subprocess, sys, os
if importlib.util.find_spec('qfin') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','-q','https://github.com/albertandking/quant-investing.git'], check=False)
        os.chdir('quant-investing')
        subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], check=False)
        subprocess.run([sys.executable,'scripts/make_sample_data.py'], check=False)


## 15.3 向量化回测（T+1：signal.shift(1)）

In [ ]:
from qfin import load_close, sma, sharpe_ratio, annualized_return, max_drawdown

px = load_close()['STK001']
ret = px.pct_change().fillna(0)
signal = (sma(px, 5) > sma(px, 20)).astype(int)
position = signal.shift(1).fillna(0)
strat = position * ret
print(f'年化收益：{annualized_return(strat):.2%}，夏普：{sharpe_ratio(strat):.2f}')

## 15.4 前视偏差：凭空多出的收益

In [ ]:
strat_correct = signal.shift(1).fillna(0) * ret
strat_look = signal * ret
print(f'无前视  年化 {annualized_return(strat_correct):.2%}  夏普 {sharpe_ratio(strat_correct):.2f}')
print(f'有前视  年化 {annualized_return(strat_look):.2%}  夏普 {sharpe_ratio(strat_look):.2f}')
print(f'前视溢价（年化差）：{annualized_return(strat_look)-annualized_return(strat_correct):+.2%}')

## 15.5 交易成本与换手

In [ ]:
turnover = position.diff().abs().fillna(0)
print(f'年均单边换手：{turnover.sum()/(len(turnover)/252):.1f} 次')
for c in [0.0, 0.0005, 0.001, 0.002]:
    net = strat - turnover * c
    print(f'单边成本 {c:.2%}: 年化 {annualized_return(net):+.2%}  夏普 {sharpe_ratio(net):.2f}')

## 15.7 数据窥探：100 个随机策略的最优夏普

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
best = max(sharpe_ratio(rng.integers(0, 2, len(ret)) * ret) for _ in range(100))
print(f'真实均线策略夏普：{sharpe_ratio(strat):.2f}')
print(f'100 个随机策略最优夏普：{best:.2f}  ← 比真策略还高，纯属运气')

## 15.7 稳健性：参数扫描

In [ ]:
for f, s in [(5,20),(10,30),(10,60),(20,60)]:
    sig = (sma(px,f) > sma(px,s)).astype(int).shift(1).fillna(0)
    print(f'窗口({f},{s})  夏普 {sharpe_ratio(sig*ret):.2f}')